### Imports

In [1]:
import os
import torch
import torchvision.transforms as T
import pandas as pd
from PIL import Image
from datasets import load_dataset

c:\Users\mohal\miniconda3\envs\INF265\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Parsing

## Setup
We want all our images to be 224 x 224 with 8 frames each. In other words our tensor would be something like 8 x 3 x 224 x 224 (frames, chennels, height width). However, it is not optimal for storage to convert the images and save them as tensors. Therefore, we will store only a reference in our metadata and load the images on demand, applying the transformations inside the dataset class.

In [2]:
SAVE_PATH = "../data/dataset/"       # path to save the data after processing
rows = []

## UTKFace data (positive examples)

In [3]:
# function to extract the age label
def parse_utkface_label(filename):
     # the first part of the name of the utkface data is the age
    return int(filename.split("_")[0])

### create samples
The UTKFace data were downloaded locally beforehand, so label parsing is done manually.

In [4]:
utkface_path = "../data/UTKFace/"

for filename in os.listdir(utkface_path):
    age = parse_utkface_label(filename)

    rows.append({
        "frames": os.path.join(utkface_path, filename),
        "person": 1,
        "age": age
    })

## Places365 data (negative examples)

In [5]:
# function to save the images
def saveImage(img, folder, name):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, name)
    img.save(path)

### create samples
Here hugging face is used to access the Places365 data. ``streaming=True`` is used so there won't be need to download the entire datset.

In [ ]:
places365   = load_dataset("Andron00e/Places365-custom", split="train", streaming=True)
n_examples  = 2000
places365_path = f"../data/Places365_{n_examples//1000}k/"

for i, data in enumerate(places365):
    if i >= n_examples:
        break

    img = data["image"].convert("RGB")
    filename = f"places_{i:06d}.jpg"
    
    # save the image
    saveImage(img, places365_path, filename)
    # add to df rows
    rows.append({
        "frames": os.path.join(places365_path, filename),
        "person": 0,
        "age": -1
    })

## Metadata

In [7]:
pd.DataFrame(rows).to_csv(os.path.join(SAVE_PATH, "metadata.csv"), index=False)